# ames-housing-mlp

_The notebook used to train this model is available on [GitHub](https://github.com/DonaldKellett/my-ascend-notebooks/blob/64597dd39911e400f5e9b810c01d9f91a94300ca/orangepiaipro-20t/08-d2l-mindspore-ch5-multilayer-perceptrons/04-predicting-house-prices-on-kaggle.ipynb)._

An MLP model trained with MindSpore 2.8.0 + CANN 8.5.0 on the [Ames Housing Dataset](https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset) with $k$-fold cross validation \($k = 10$\) and the following architecture.

1. Hidden layer with $288$ input channels and $64$ output channels, ReLU activation
1. Dropout layer with $p = 0.2$
1. Linear layer with $64$ input channels and $1$ output channel

Related links:

1. [5.7. Predicting House Prices on Kaggle](https://d2l.ai/chapter_multilayer-perceptrons/kaggle-house-price.html) in the D2L courseware
1. [House Prices: Advanced Regression Techniques | Kaggle](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) competition

This notebook demonstrates how to load the trained ONNX model for inference.

## Dependencies

ONNX Runtime 1.25.0 with GPU acceleration

In [1]:
%pip install onnxruntime-gpu==1.25.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.3/270.3 MB 6.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


## Serving with ONNX Runtime

In [2]:
import os

model_path = '/kaggle/input/models/donaldsebleung/ames-housing-mlp/onnx/default/2/ames-housing-mlp.onnx'
os.path.isfile(model_path)

True

In [3]:
import onnxruntime as ort

available_providers = ort.get_available_providers()
available_providers

['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']

In [4]:
session = ort.InferenceSession(model_path, providers=['CUDAExecutionProvider'])
session.get_providers()

['CUDAExecutionProvider', 'CPUExecutionProvider']

## Downloading the dataset

We'll download a copy of the Ames housing dataset from D2L-owned S3 bucket(s).

In [5]:
import os

dataset_dir = 'data/ames/'
os.makedirs(dataset_dir, exist_ok=True)

In [6]:
import urllib.request

train_ds_link = 'http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_train.csv'
test_ds_link = 'http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_test.csv'
train_ds_path = os.path.join(dataset_dir, 'kaggle_house_pred_train.csv')
test_ds_path = os.path.join(dataset_dir, 'kaggle_house_pred_test.csv')

with urllib.request.urlopen(train_ds_link) as response:
    with open(train_ds_path, 'wb') as file:
        file.write(response.read())

with urllib.request.urlopen(test_ds_link) as response:
    with open(test_ds_path, 'wb') as file:
        file.write(response.read())

## Loading the dataset with Pandas

In [7]:
import pandas as pd

train_df = pd.read_csv(train_ds_path)
test_df = pd.read_csv(test_ds_path)
train_df.shape, test_df.shape

((1460, 81), (1459, 80))

## Preprocessing the features and labels for our model

The original dataset has $1460$ training samples and $1459$ holdout samples with $80$ input features and $1$ output label. Let's apply the following transformations before feeding it to our model.

For the features:

1. Remove the redundant ID column
1. Compute the mean for each column of numerical data, skipping NA values from the mean computation
1. Replace NA values in numerical data with the mean value from their corresponding column
1. Apply StandardScaler to numerical columns to have zero mean and unit variance
1. Apply one-hot encoding to categorical \(non-numerical\) columns with the following rules:
    1. Treat NA values as its own distinct category
    1. Remove the 1st category from the generated one-hot encoding to avoid collinearity

For the labels:

1. Apply `log1p` transformation to the sale price
1. Apply StandardScaler to \(1\) so the transformed labels have zero mean and unit variance

After pre-processing and transformation, we have:

1. 1460 training samples
1. 288 input features - generated from the original 80 input features after applying one-hot encoding
1. 1 output label corresponding to the house price

In [8]:
from sklearn.base import BaseEstimator, TransformerMixin

class LogStandardScaler(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.scaler = StandardScaler()

    def fit(self, y):
        y_log1p = np.log1p(y)
        self.scaler.fit(y_log1p)
        return self

    def transform(self, y):
        y_log1p = np.log1p(y)
        y_scaled = self.scaler.transform(y_log1p)
        return y_scaled

    def inverse_transform(self, y_scaled):
        y_log1p = self.scaler.inverse_transform(y_scaled)
        y = np.expm1(y_log1p)
        return y

In [9]:
import numpy as np

train_onehot_columns = None

def preprocess_features(X_dataframe, is_train=False):
    global train_onehot_columns
    
    """
    Split input features by numeric and non-numeric columns
    """
    X_dataframe_numeric = X_dataframe.select_dtypes(include=['number'])
    X_dataframe_categoric = X_dataframe.select_dtypes(exclude=['number'])

    """
    For numeric data, fill NA values with the column mean
    """
    X_dataframe_numeric = X_dataframe_numeric.fillna(X_dataframe_numeric.mean(numeric_only=True))

    """
    For non-numeric data, apply one-hot encoding with the caveats below.
    1. Discard the 1st category to avoid collinearity
    2. Treat NA values as its own distinct category
    3. For training data, record the resulting columns from one-hot encoding
    4. For test data, reindex based on (3) to ensure training and test data have the same number of input features
    """
    X_dataframe_categoric = pd.get_dummies(X_dataframe_categoric, drop_first=True, dummy_na=True)
    if is_train:
        train_onehot_columns = X_dataframe_categoric.columns
    else:
        X_dataframe_categoric = X_dataframe_categoric.reindex(columns=train_onehot_columns, fill_value=0)

    """
    Convert numeric and non-numeric input features to separate Numpy arrays
    """
    X_numeric = X_dataframe_numeric.to_numpy()
    X_categoric = X_dataframe_categoric.to_numpy()

    """
    Return the numeric and non-numeric input features as separate ndarrays
    """
    return X_numeric, X_categoric

In [10]:
X_train_df = train_df.iloc[:, :-1]
y_train_df = train_df.iloc[:, -1]
X_train_numeric, X_train_categoric = preprocess_features(X_dataframe=X_train_df, is_train=True)
y_train = y_train_df.to_numpy().reshape(-1, 1)
X_train_numeric.shape, X_train_categoric.shape, y_train.shape

((1460, 37), (1460, 251), (1460, 1))

Our pre-processed data has 37 numeric features and 251 non-numeric features for a total of 288 input features.

In [11]:
from sklearn.preprocessing import StandardScaler

feature_scaler, label_scaler = StandardScaler(), LogStandardScaler()
X_train_numeric_scaled = feature_scaler.fit_transform(X_train_numeric)
y_train_scaled = label_scaler.fit_transform(y_train)
X_train_scaled = np.concatenate((X_train_numeric_scaled, X_train_categoric), axis=1)
X_train_scaled.shape, y_train_scaled.shape

((1460, 288), (1460, 1))

## Inferencing with ONNX Runtime

Let's take the first 5 samples from our training set and use the model to predict the house prices. We'll need to use `inverse_transform` to recover meaningful prediction values. Let's also calculate the percentage difference between the actual and predicted house prices.

In [12]:
X_train_scaled_samples = X_train_scaled[:5]
y_train_samples = y_train[:5]
X_train_scaled_samples.shape, y_train_samples.shape

((5, 288), (5, 1))

In [13]:
input_name = session.get_inputs()[0].name
input_name

'feature'

In [14]:
for i in range(5):
    y_hat_scaled = session.run(None, {input_name: X_train_scaled_samples[i:i+1]})
    y_hat_scaled = y_hat_scaled[0].astype(np.float32)
    y_hat = label_scaler.inverse_transform(y_hat_scaled)
    y_hat = y_hat.item()
    y = y_train_samples[i].item()
    print(f'Is: USD${y:.2f}, Got: USD${y_hat:.2f}, Error: {100 * (y_hat - y) / y:.4f}%')

Is: USD$208500.00, Got: USD$213185.62, Error: 2.2473%
Is: USD$181500.00, Got: USD$184470.61, Error: 1.6367%
Is: USD$223500.00, Got: USD$219898.14, Error: -1.6116%
Is: USD$140000.00, Got: USD$174989.16, Error: 24.9923%
Is: USD$250000.00, Got: USD$299935.84, Error: 19.9743%


As seen from above, the prediction error is around $15\%$ on average - not too bad!